In [1]:
# CELL 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import lightgbm as lgb
from scipy import stats
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Paths
QWS_PATH  = '/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/Dataset/Scheme_A/'
VBFC_PATH = '/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/exp_ QWS to VFBC synthetic transferability/Synthetic_VBFC_Dataset/'
SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/Ranking_Selection/exp_ QWS to VFBC synthetic transferability/'

import os
os.makedirs(SAVE_PATH, exist_ok=True)

Mounted at /content/drive


In [2]:
# CELL 2: Define columns and metric functions

NORM_COLS = [
    'Response_Time_norm',
    'Availability_norm',
    'Throughput_norm',
    'Reliability_norm',
    'Latency_norm'
]

def compute_mae(true_ranks, pred_ranks):
    return np.mean(np.abs(np.array(true_ranks) - np.array(pred_ranks)))

def compute_top1(true_ranks, pred_ranks):
    return int(np.argmin(pred_ranks) == np.argmin(true_ranks))

def compute_spearman(true_ranks, pred_ranks):
    corr, _ = spearmanr(true_ranks, pred_ranks)
    return corr

def compute_ndcg(true_ranks, pred_ranks, k=10):
    n = len(true_ranks)
    true_ranks = np.array(true_ranks)
    pred_ranks = np.array(pred_ranks)
    relevance  = (n + 1) - true_ranks
    pred_order = np.argsort(pred_ranks)
    sorted_rel = relevance[pred_order]
    dcg  = sum(sorted_rel[i] / np.log2(i + 2) for i in range(min(k, n)))
    idcg = sum(np.sort(relevance)[::-1][i] / np.log2(i + 2) for i in range(min(k, n)))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_method(test_df, pred_col):
    mae_l, top1_l, sp_l, ndcg_l = [], [], [], []
    for lid in test_df['list_id'].unique():
        g = test_df[test_df['list_id'] == lid]
        tr = g['list_rank'].values
        pr = g[pred_col].values
        mae_l.append(compute_mae(tr, pr))
        top1_l.append(compute_top1(tr, pr))
        sp_l.append(compute_spearman(tr, pr))
        ndcg_l.append(compute_ndcg(tr, pr))
    return {
        'MAE':      round(np.mean(mae_l), 4),
        'Top-1':    round(np.mean(top1_l) * 100, 2),
        'Spearman': round(np.mean(sp_l), 4),
        'NDCG':     round(np.mean(ndcg_l), 4)
    }

In [3]:
# CELL 3: LambdaMART function with seed support

def run_lambdamart(train_df, val_df, test_df, seed=42):
    def prep(df):
        X, y, g = [], [], []
        for lid in df['list_id'].unique():
            grp = df[df['list_id'] == lid]
            X.append(grp[NORM_COLS].values)
            y.append((11 - grp['list_rank']).values)
            g.append(len(grp))

        return np.vstack(X), np.concatenate(y), g

    X_tr, y_tr, g_tr = prep(train_df)
    X_vl, y_vl, g_vl = prep(val_df)
    X_te, _,    _    = prep(test_df)

    dtrain = lgb.Dataset(X_tr, label=y_tr, group=g_tr)
    dval   = lgb.Dataset(X_vl, label=y_vl, group=g_vl)

    params = {
        'objective':        'lambdarank',
        'metric':           'ndcg',
        'ndcg_eval_at':     [1, 5, 10],
        'learning_rate':    0.05,
        'num_leaves':       31,
        'verbose':          -1,
        'seed':             seed,
        'bagging_fraction': 0.9,
        'bagging_freq':     5,
        'feature_fraction': 0.9,
    }
    model = lgb.train(
        params, dtrain, num_boost_round=200,
        valid_sets=[dval],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
    )
    scores = model.predict(X_te)
    test_df = test_df.copy()
    test_df['lgb_score'] = scores
    ranks = []
    for lid in test_df['list_id'].unique():
        g = test_df[test_df['list_id'] == lid]
        r = g['lgb_score'].rank(ascending=False, method='min').astype(int)
        ranks.extend(r.values)
    test_df['lgb_rank'] = ranks
    return evaluate_method(test_df, 'lgb_rank')

def aggregate_runs(runs):
    result = {}
    for metric in runs[0]:
        scores = [r[metric] for r in runs]
        mean = np.mean(scores)
        sem = stats.sem(scores)
        if np.isnan(sem) or sem == 0:
            margin = 0.0
        else:
            ci = stats.t.interval(0.95, len(scores)-1, loc=mean, scale=sem)
            margin = round(mean - ci[0], 4)
        result[metric] = round(mean, 4)
        result[f'{metric}_ci'] = margin
    return result


In [4]:
# CELL 4: Load all datasets

# QWS datasets
qws_train = pd.read_csv(QWS_PATH + 'obj3_train_A.csv')
qws_val   = pd.read_csv(QWS_PATH + 'obj3_val_A.csv')
qws_test  = pd.read_csv(QWS_PATH + 'obj3_test_A.csv')

# VBFC synthetic datasets
vbfc_train = pd.read_csv(VBFC_PATH + 'vbfc_train.csv')
vbfc_val   = pd.read_csv(VBFC_PATH + 'vbfc_val.csv')
vbfc_test  = pd.read_csv(VBFC_PATH + 'vbfc_test.csv')

print(f'QWS  — train: {qws_train["list_id"].nunique()} lists, val: {qws_val["list_id"].nunique()}, test: {qws_test["list_id"].nunique()}')
print(f'VBFC — train: {vbfc_train["list_id"].nunique()} lists, val: {vbfc_val["list_id"].nunique()}, test: {vbfc_test["list_id"].nunique()}')

QWS  — train: 2005 lists, val: 251, test: 251
VBFC — train: 2005 lists, val: 251, test: 251


In [6]:
# CELL 5: Run both transfer experiments with 5 seeds

seeds = [42, 123, 456, 789, 1011]

# Experiment 1 — QWS -> VBFC
print('='*60)
print('Experiment 1: Train on QWS, Test on VBFC-synthetic')
print('='*60)
exp1_runs = []
for s in seeds:
    print(f'  Seed {s}...')
    res = run_lambdamart(qws_train, qws_val, vbfc_test, seed=s)
    exp1_runs.append(res)
exp1_results = aggregate_runs(exp1_runs)
print(f'Results: {exp1_results}')

# Experiment 2 — VBFC -> QWS
print('='*60)
print('Experiment 2: Train on VBFC-synthetic, Test on QWS')
print('='*60)
exp2_runs = []
for s in seeds:
    print(f'  Seed {s}...')
    res = run_lambdamart(vbfc_train, vbfc_val, qws_test, seed=s)
    exp2_runs.append(res)
exp2_results = aggregate_runs(exp2_runs)
print(f'Results: {exp2_results}')


Experiment 1: Train on QWS, Test on VBFC-synthetic
  Seed 42...
Training until validation scores don't improve for 50 rounds
[50]	valid_0's ndcg@1: 0.970153	valid_0's ndcg@5: 0.982716	valid_0's ndcg@10: 0.986323
[100]	valid_0's ndcg@1: 0.97115	valid_0's ndcg@5: 0.985634	valid_0's ndcg@10: 0.988422
Early stopping, best iteration is:
[53]	valid_0's ndcg@1: 0.975138	valid_0's ndcg@5: 0.984086	valid_0's ndcg@10: 0.987658
  Seed 123...
Training until validation scores don't improve for 50 rounds
[50]	valid_0's ndcg@1: 0.969093	valid_0's ndcg@5: 0.983404	valid_0's ndcg@10: 0.98678
[100]	valid_0's ndcg@1: 0.975075	valid_0's ndcg@5: 0.986369	valid_0's ndcg@10: 0.988917
Early stopping, best iteration is:
[61]	valid_0's ndcg@1: 0.979063	valid_0's ndcg@5: 0.986131	valid_0's ndcg@10: 0.989279
  Seed 456...
Training until validation scores don't improve for 50 rounds
[50]	valid_0's ndcg@1: 0.97115	valid_0's ndcg@5: 0.98348	valid_0's ndcg@10: 0.986838
[100]	valid_0's ndcg@1: 0.969093	valid_0's ndcg@

In [7]:
# CELL 6: Build results table and save

results_table = pd.DataFrame([
    {
        'Experiment': 'QWS -> VBFC-synthetic',
        'Train': 'QWS',
        'Test': 'VBFC-synthetic',
        'MAE': exp1_results['MAE'],
        'MAE_ci': exp1_results['MAE_ci'],
        'Top-1': exp1_results['Top-1'],
        'Top-1_ci': exp1_results['Top-1_ci'],
        'Spearman': exp1_results['Spearman'],
        'Spearman_ci': exp1_results['Spearman_ci'],
        'NDCG': exp1_results['NDCG'],
        'NDCG_ci': exp1_results['NDCG_ci'],
    },
    {
        'Experiment': 'VBFC-synthetic -> QWS',
        'Train': 'VBFC-synthetic',
        'Test': 'QWS',
        'MAE': exp2_results['MAE'],
        'MAE_ci': exp2_results['MAE_ci'],
        'Top-1': exp2_results['Top-1'],
        'Top-1_ci': exp2_results['Top-1_ci'],
        'Spearman': exp2_results['Spearman'],
        'Spearman_ci': exp2_results['Spearman_ci'],
        'NDCG': exp2_results['NDCG'],
        'NDCG_ci': exp2_results['NDCG_ci'],
    }
])

print('='*60)
print('TRANSFER EXPERIMENT RESULTS')
print('='*60)
print(results_table.to_string())

results_table.to_csv(SAVE_PATH + 'transfer_experiment_results.csv', index=False)
print(f'\nSaved to: {SAVE_PATH}transfer_experiment_results.csv')

TRANSFER EXPERIMENT RESULTS
              Experiment           Train            Test     MAE  MAE_ci   Top-1  Top-1_ci  Spearman  Spearman_ci    NDCG  NDCG_ci
0  QWS -> VBFC-synthetic             QWS  VBFC-synthetic  1.2547  0.0078  39.842    1.8492    0.8055       0.0028  0.9566   0.0010
1  VBFC-synthetic -> QWS  VBFC-synthetic             QWS  1.2178  0.0653  69.642    4.7099    0.8047       0.0169  0.9670   0.0032

Saved to: /content/drive/MyDrive/Colab Notebooks/Ranking_Selection/exp_ QWS to VFBC synthetic transferability/transfer_experiment_results.csv
